# Constellation Boundaries

Plots the IAU constellation boundaries (from `resources/*.txt`) using Plotly.

Each file contains the vertices of a constellation's boundary polygon as `RA (hh mm ss.ssss) | Dec (deg) | CODE`.

In [1]:
from pathlib import Path

import pandas as pd
import plotly.graph_objects as go

RESOURCES_DIR = Path("resources")

In [2]:
def ra_to_hours(ra_str: str) -> float:
    """Convert 'hh mm ss.ssss' right ascension to decimal hours."""
    h, m, s = ra_str.split()
    return float(h) + float(m) / 60 + float(s) / 3600


def load_boundaries(resources_dir: Path) -> pd.DataFrame:
    records = []
    for path in sorted(resources_dir.glob("*.txt")):
        if path.name == "figures.txt":
            continue
        with open(path) as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                ra_str, dec_str, code = line.split("|")
                records.append(
                    {
                        "ra_h": ra_to_hours(ra_str.strip()),
                        "dec_deg": float(dec_str.strip()),
                        "constellation": code.strip(),
                    }
                )
    return pd.DataFrame(records)


boundaries = load_boundaries(RESOURCES_DIR)
boundaries.head()

,ra_h,dec_deg,constellation
0,22.964354,35.168236,AND
1,22.956190,53.168030,AND
2,23.430193,53.187004,AND
3,23.431045,50.687019,AND
4,23.684704,50.692913,AND


In [3]:
def boundary_trace(name: str, group: pd.DataFrame) -> go.Scatter:
    """Build a closed-polygon line trace, splitting segments that wrap across RA 0h/24h."""
    ra = group["ra_h"].tolist() + [group["ra_h"].iloc[0]]
    dec = group["dec_deg"].tolist() + [group["dec_deg"].iloc[0]]

    ra_out, dec_out = [ra[0]], [dec[0]]
    for prev_ra, cur_ra, cur_dec in zip(ra, ra[1:], dec[1:]):
        if abs(cur_ra - prev_ra) > 12:  # RA wraparound, break the line
            ra_out.append(None)
            dec_out.append(None)
        ra_out.append(cur_ra)
        dec_out.append(cur_dec)

    return go.Scatter(
        x=ra_out,
        y=dec_out,
        mode="lines",
        name=name,
        line=dict(width=1, color="grey"),
        hoverinfo="text",
        hovertext=name,
        showlegend=False,
    )


fig = go.Figure(
    data=[
        boundary_trace(name, group)
        for name, group in boundaries.groupby("constellation", sort=False)
    ]
)
fig.update_layout(
    title="IAU Constellation Boundaries",
    xaxis=dict(title="Right Ascension (h)", autorange="reversed", range=[24, 0], dtick=2),
    yaxis=dict(title="Declination (°)", range=[-90, 90], dtick=15),
    width=1200,
    height=700,
    template="plotly_dark",
    paper_bgcolor="black",
    plot_bgcolor="black",
)

## Constellation Figures

Adds the stick-figure lines connecting stars (from `resources/figures.txt`, `conline` records) on top of the boundaries.

In [4]:
def load_figures(resources_dir: Path) -> pd.DataFrame:
    """Parse the 'conline' records: pairwise star connections for each constellation figure."""
    records = []
    with open(resources_dir / "figures.txt") as f:
        for line in f:
            if not line.startswith("conline"):
                continue
            fields = line.split()
            records.append(
                {
                    "constellation": fields[1],
                    "thickness": int(fields[2]),
                    "ra1_h": float(fields[9]) / 15,
                    "dec1_deg": float(fields[10]),
                    "ra2_h": float(fields[19]) / 15,
                    "dec2_deg": float(fields[20]),
                }
            )
    return pd.DataFrame(records)


figures = load_figures(RESOURCES_DIR)
figures.head()

,constellation,thickness,ra1_h,dec1_deg,ra2_h,dec2_deg
0,And,1,1.633211,48.628212,1.158368,47.241794
1,And,1,1.158368,47.241794,0.830235,41.078910
2,And,1,0.830235,41.078910,0.945891,38.499334
3,And,1,0.945891,38.499334,1.162201,35.620558
4,And,1,1.162201,35.620558,0.614680,33.719344


In [5]:
def figure_trace(row: pd.Series) -> go.Scatter:
    """Build a single star-to-star connection, shifted to avoid spurious RA 0h/24h wraparound."""
    ra1, ra2 = row["ra1_h"], row["ra2_h"]
    if ra2 - ra1 > 12:
        ra2 -= 24
    elif ra1 - ra2 > 12:
        ra2 += 24

    return go.Scatter(
        x=[ra1, ra2],
        y=[row["dec1_deg"], row["dec2_deg"]],
        mode="lines",
        name=row["constellation"],
        line=dict(width=row["thickness"], color="#00ff00"),
        hoverinfo="text",
        hovertext=row["constellation"],
        showlegend=False,
    )


fig.add_traces([figure_trace(row) for _, row in figures.iterrows()])
fig.update_layout(title="IAU Constellation Boundaries and Figures")

## Bright Stars

Overplots stars from the Yale Bright Star Catalogue, 5th ed. (YBSC5), whose entries are embedded as `star` records in `resources/figures.txt`. Marker size scales with brightness — a *smaller* magnitude means a *brighter* star, so lower-magnitude stars get larger markers, using the same relative-flux scaling (`10^(-0.2*mag)`) astronomers use for star-chart symbol sizes.

In [6]:
MIN_MARKER_SIZE = 1
MAX_MARKER_SIZE = 6


def load_bright_stars(resources_dir: Path) -> pd.DataFrame:
    """Parse the YBSC5 'star' records: HR catalogue index, position, and magnitude."""
    records = []
    with open(resources_dir / "figures.txt") as f:
        for line in f:
            if not line.startswith("star"):
                continue
            fields = line.split()
            mag = float(fields[18])
            if mag == 99:  # sentinel for missing magnitude
                continue
            records.append(
                {
                    "hr": fields[2],
                    "constellation": fields[5],
                    "name": fields[13],
                    "ra_deg": float(fields[15]),
                    "dec_deg": float(fields[16]),
                    "mag": mag,
                }
            )
    return pd.DataFrame(records)


bright_stars = load_bright_stars(RESOURCES_DIR)
bright_stars.head()

,hr,constellation,name,ra_deg,dec_deg,mag
0,1,And,1,1.290659,45.229031,6.710
1,2,Psc,2,1.265928,-0.503036,6.298
2,3,Psc,abbrev,1.333922,-5.707620,4.610
3,4,Peg,86,1.424840,13.396268,5.537
4,5,Cas,5,1.565892,58.436728,4.329


In [7]:
rel_flux = 10 ** (-0.2 * bright_stars["mag"])  # brightness ~ sqrt(flux), so area scales with flux
marker_size = MIN_MARKER_SIZE + (MAX_MARKER_SIZE - MIN_MARKER_SIZE) * (
    rel_flux - rel_flux.min()
) / (rel_flux.max() - rel_flux.min())

star_trace = go.Scatter(
    x=bright_stars["ra_deg"] / 15,
    y=bright_stars["dec_deg"],
    mode="markers",
    marker=dict(size=marker_size, color="white", line=dict(width=0.5, color="dimgray")),
    name="Bright stars",
    hoverinfo="text",
    hovertext=[
        f"HR {hr} ({name}) {con}<br>mag {m:.2f}"
        for hr, name, con, m in zip(
            bright_stars["hr"], bright_stars["name"], bright_stars["constellation"], bright_stars["mag"]
        )
    ],
    showlegend=False,
)

fig.add_trace(star_trace)
fig.update_layout(title="IAU Constellation Boundaries, Figures, and Bright Stars")
fig.show()